# Day 3 Conversational ChatBot!

In [19]:
from openai import OpenAI
import gradio as gr
from ollama import Client


In [42]:
# setup
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL_OLLAMA = 'llama3.2:3b'
GIMINI_BASE_URL= "http://localhost:11434"
MODEL_GEMMNI = "gemma3:270m"
openai = OpenAI()
ollama = OpenAI(base_url=OLLAMA_BASE_URL,api_key='ollama') 
client=Client(host=GIMINI_BASE_URL)

In [3]:
system_message = "You are a helpful assistant"

### Wrighting a callback function which perform task this function get called by gradio

In [4]:
def chat(message,history):
    return "banana"

In [5]:
# create the chatbot ui
view_chatbot=gr.ChatInterface(
    fn=chat,
    # type='messages'
)
view_chatbot.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [6]:
def chat(message,history):
    return f" you said {message} and the history is {history}"

In [7]:
gr.ChatInterface(
    fn=chat
).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Now use the Model to get the response 

In [10]:
# create chat function which talk with model
# History means when we talk the previous message are in history like blow
# you said my name is faizan and the history is 
# [{'role': 'user', 'metadata': None, 'content': [{'text': 'hello chat bot', 'type': 'text'}], 'options': None}, 
# {'role': 'assistant', 'metadata': None, 'content': [{'text': 'you said hello chat bot and the history is []', 'type': 'text'}], 'options': None}]
# history has more data which is not used so we get required history data using blow line


def chat(message,history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages=[{'role':'system','content':system_message}] + history + [{'role':'user','content':message}]
    response=ollama.chat.completions.create(model=MODEL_OLLAMA,messages=messages)
    return response.choices[0].message.content


In [11]:
gr.ChatInterface(
    fn=chat
).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## Now lats make our function as stream 


In [23]:
# using ollama model
def chat_ollama(message,history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages=[{'role':'system','content':system_message}] + history + [{'role':'user','content':message}]
    stream= ollama.chat.completions.create(model=MODEL_OLLAMA,messages=messages, stream=True)
    result =''
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
    yield result

#using gimini model

def chat_gemini(message,history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages=[{'role':'system','content':system_message}] + history + [{'role':'user','content':message}]
    stream= client.chat(model=MODEL_OLLAMA,messages=messages, stream=True)
    result =''
    for chunk in stream:
        result += chunk['message']['content'] or ""
    yield result

In [24]:
gr.ChatInterface(
    fn=chat_ollama
).launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


## lest update system promt variable 


In [25]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [ ]:
gr.ChatInterface(
    fn=chat_gemini
).launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [27]:
# Adding more text in system prompt 
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [28]:

print(system_message)
gr.ChatInterface(
    fn=chat_ollama
    ).launch()

You are a helpful assistant in a clothes store. You should try to gently encourage the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. For example, if the customer says 'I'm looking to buy a hat', you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'Encourage the customer to buy hats if they are unsure what to get.
If the customer asks for shoes, you should respond that shoes are not on sale today, but remind the customer to look at hats!
If the customer asks for shoes, you should respond that shoes are not on sale today, but remind the customer to look at hats!
* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


## chat function with different system message

In [ ]:


def chat(message,history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if "belt" in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]
    stream = ollama.chat.completions.create(model=MODEL_OLLAMA, messages=messages, stream=True)
    response =''
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
    yield response   




In [35]:
gr.ChatInterface(
    fn=chat
    ).launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


## some different system messages 

In [36]:
new_system_message = """
You are a travel agent.
Suggest destinations based on the customer's interests and budget.
Ask one follow-up question when needed.
Keep responses under 50 words.
"""

In [53]:

model = 'qwen2.5:3b'
def chat(message,history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = ollama.chat.completions.create(model=model, messages=messages, stream=True)
    response =''
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
    yield response   


In [54]:
gr.ChatInterface(
    fn=chat
).launch()

* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.


In [41]:
!ollama pull llama3.2:3b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01

In [47]:

def yes_man(message, history):
    if message.endswith("?"):
        return "Yes"
    else:
        return "Ask me anything!"

gr.ChatInterface(
    yes_man,
    chatbot=gr.Chatbot(height=300),
    textbox=gr.Textbox(placeholder="Ask me a yes or no question", container=False, scale=7),
    title="Yes Man",
    description="Ask Yes Man any question",
    examples=["Hello", "Am I cool?", "Are tomatoes vegetables?"],
    cache_examples=True,
).launch(theme="ocean")

* Running on local URL:  http://127.0.0.1:7878
Caching examples at: 'c:\Users\LOG IN\Desktop\llm_engineering\week2\community-contributions\faizan_hamid\.gradio\cached_examples\511'
* To create a public link, set `share=True` in `launch()`.


In [48]:
!ollama pull qwen2.5:3b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠴ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling 5ee4f07cdb9b:   0% ▕                  ▏ 151 KB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:   0% ▕                  ▏ 1.7 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:   0% ▕                  ▏ 2.4 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:   0% ▕                  ▏ 4.3 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:   0% ▕                  ▏ 6.4 MB/1.9 GB                  pulling manifest 
pulling 5ee4f07cdb9b:   0% ▕                  ▏ 7.4 MB/1.9 GB                  

In [49]:
!ollama list

NAME               ID              SIZE      MODIFIED       
qwen2.5:3b         357c53fb659c    1.9 GB    32 seconds ago    
llama3.2:3b        a80c4f17acd5    2.0 GB    25 minutes ago    
llama3.2:1b        baf6a787fdff    1.3 GB    7 days ago        
llama3.2:latest    a80c4f17acd5    2.0 GB    8 days ago        
gemma3:270m        e7d36fb2c3b3    291 MB    10 days ago       


In [55]:
!ollama rm gemma3:270m

deleted 'gemma3:270m'


⠙ ⠹ ⠸ ⠼ ⠴ ⠴ ⠧ ⠧ ⠇ ⠏ 


In [56]:
!ollama rm llama3.2:1b

deleted 'llama3.2:1b'


⠙ ⠹ ⠹ ⠼ ⠴ ⠴ ⠧ 


In [57]:
!ollama rm llama3.2:latest

deleted 'llama3.2:latest'


⠙ ⠙ ⠸ ⠼ ⠴ ⠴ ⠧ ⠇ ⠇ ⠋ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ 


In [58]:
!ollama list


NAME           ID              SIZE      MODIFIED       
qwen2.5:3b     357c53fb659c    1.9 GB    11 minutes ago    
llama3.2:3b    a80c4f17acd5    2.0 GB    36 minutes ago    
